# `ty` — Colab T4 local inference server (one URL, model picked by request)

Runs TWO real, OpenAI-compatible `llama-server` processes on Colab's free T4 GPU
(one per model), puts a small router in front of them so your app only ever talks to
**one URL**, and exposes that single router publicly via a Cloudflare Tunnel (no
signup needed). Which model actually answers is decided entirely by the `model`
field your `ty` app sends in each request — switching models later is just changing
`AI_MODEL` in `.env`, never the URL.

**Models used (verified sources, same ones Thuki itself ships):**
- `qwen3.5-9b` — ~5.7GB, better quality, vision-capable. **Default.**
- `llama-3.2-3b` — tiny, ~2.0GB, fastest, good fallback if you want more headroom.

Both fit on a T4 (16GB VRAM) at the same time with headroom to spare.

**Before you run anything:** Runtime -> Change runtime type -> T4 GPU.

**Honest limits, read this once:** Colab free sessions disconnect after ~90 min idle
and have a hard ceiling (often under 12h). Google's Colab terms do not permit using
it as a persistent always-on server for external traffic — treat this notebook as a
**temporary experiment / learning exercise**, not your daily-use backend. The tunnel
URL changes every time you restart this notebook, so you'll need to update `.env`
again after any disconnect.


## 1. Confirm the T4 GPU is attached

In [ ]:
!nvidia-smi


## 2. Build `llama-server` with CUDA support

Building from source (rather than a prebuilt release zip) is slower (~5-8 min) but
far more robust: prebuilt release asset names and CUDA versions change across
llama.cpp releases and can silently stop matching Colab's installed CUDA toolkit.
`-DGGML_CUDA=ON` is the current, verified-correct flag that offloads inference to
the T4 (checked directly against llama.cpp's own build docs); without it this runs
on CPU and is unusably slow.


In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
%cd llama.cpp
!cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
!cmake --build build --config Release -j $(nproc) --target llama-server


## 3. Download both GGUF models

Exact repo/filename pairs, pulled straight from Thuki's own curated, SHA-256-pinned
model registry (`src-tauri/src/models/registry.rs`) — verified real sources, not
guesses.


In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
import os

os.makedirs("/content/models", exist_ok=True)

qwen_9b_path = hf_hub_download(
    repo_id="unsloth/Qwen3.5-9B-GGUF",
    filename="Qwen3.5-9B-Q4_K_M.gguf",
    revision="3885219b6810b007914f3a7950a8d1b469d598a5",
    local_dir="/content/models",
)

llama_3b_path = hf_hub_download(
    repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
    filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    revision="5ab33fa94d1d04e903623ae72c95d1696f09f9e8",
    local_dir="/content/models",
)

print("qwen3.5-9b   ->", qwen_9b_path)
print("llama-3.2-3b ->", llama_3b_path)


## 4. Start both model servers, each on its own internal port

These ports (`8000`, `8001`) never get exposed publicly — only the router in the
next step gets tunneled. Each `llama-server` instance only ever serves the one GGUF
it was started with; that's exactly why the router in front of them exists.


In [ ]:
import subprocess

model_backends = []  # (logical_name, port, process)

def start_backend(model_path: str, port: int, logical_name: str, ctx_size: int = 8192):
    proc = subprocess.Popen(
        [
            "/content/llama.cpp/build/bin/llama-server",
            "-m", model_path,
            "--port", str(port),
            "--host", "127.0.0.1",   # internal only, the router is the one public door
            "-c", str(ctx_size),
            "-ngl", "999",            # offload every layer to the GPU
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    model_backends.append((logical_name, port, proc))
    print(f"starting {logical_name} on port {port} (pid {proc.pid})")

start_backend(qwen_9b_path, port=8000, logical_name="qwen3.5-9b")
start_backend(llama_3b_path, port=8001, logical_name="llama-3.2-3b")


### Wait for both to actually be ready

A fixed `sleep()` here would be a race: a ~5.7GB model can take longer than any
guessed number of seconds to finish loading onto the GPU, especially on a cold
Colab instance. `llama-server` exposes a real `/health` endpoint (200 once the
model is loaded, 503 with `"Loading model"` before that) — poll it instead of
guessing.


In [ ]:
import time
import urllib.request
import urllib.error

def wait_until_ready(port: int, logical_name: str, timeout_s: int = 180):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2) as resp:
                if resp.status == 200:
                    print(f"{logical_name} is ready")
                    return
        except (urllib.error.URLError, ConnectionError):
            pass  # not up yet, or still loading (503) -> keep polling
        time.sleep(2)
    raise RuntimeError(f"{logical_name} did not become ready within {timeout_s}s")

for logical_name, port, proc in model_backends:
    wait_until_ready(port, logical_name)


## 5. The router: picks a backend by the request's `model` field

A small, standard reverse-proxy pattern — nothing clever, which is the point:
read the JSON body, look up `model` in a plain dict, forward the request to that
backend, stream the response straight back unchanged. Falls back to
`DEFAULT_MODEL` (`qwen3.5-9b`) if the field is missing or unrecognized, rather than
erroring, so a typo in `.env` degrades gracefully instead of breaking the whole app.

One shared `httpx.AsyncClient` is created once at startup and reused for every
request (the idiomatic pattern — a fresh client per request would rebuild its
connection pool every single call for no benefit). It also sets a short
**connect** timeout only: if a backend is genuinely down, the router fails fast
instead of hanging forever, but once a response starts streaming there's no
read-timeout cutting off a long answer mid-generation.


In [ ]:
router_code = '''
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
import httpx

app = FastAPI()

# logical model name -> internal backend URL. Add a line here for every
# llama-server you start above; nothing else needs to change.
MODEL_ROUTES = {
    "qwen3.5-9b": "http://127.0.0.1:8000",
    "llama-3.2-3b": "http://127.0.0.1:8001",
}
DEFAULT_MODEL = "qwen3.5-9b"

# One shared client, reused across every request; only the CONNECT phase times
# out quickly (a dead backend fails fast), the read phase never does (a long
# streamed answer is never cut off mid-generation).
http_client = httpx.AsyncClient(
    timeout=httpx.Timeout(connect=5.0, read=None, write=None, pool=None)
)


@app.on_event("shutdown")
async def shutdown_client():
    await http_client.aclose()


@app.get("/")
async def health():
    return {"status": "ok", "default_model": DEFAULT_MODEL, "available_models": list(MODEL_ROUTES.keys())}


@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    requested_model = body.get("model")
    backend_url = MODEL_ROUTES.get(requested_model, MODEL_ROUTES[DEFAULT_MODEL])

    upstream_request = http_client.build_request(
        "POST", f"{backend_url}/v1/chat/completions", json=body
    )
    upstream_response = await http_client.send(upstream_request, stream=True)

    async def relay():
        async for chunk in upstream_response.aiter_raw():
            yield chunk
        await upstream_response.aclose()

    content_type = upstream_response.headers.get("content-type", "text/event-stream")
    return StreamingResponse(relay(), media_type=content_type)
'''

with open("/content/router.py", "w") as f:
    f.write(router_code)

print("wrote /content/router.py")


In [ ]:
!pip install -q fastapi "uvicorn[standard]" httpx

router_proc = subprocess.Popen(
    ["uvicorn", "router:app", "--host", "0.0.0.0", "--port", "9000"],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f"router starting on port 9000 (pid {router_proc.pid})")

time.sleep(3)


## 6. Install `cloudflared` and tunnel ONLY the router (one URL, total)

Cloudflare's "quick tunnel" needs no account, no authtoken — it just hands you a
random `https://*.trycloudflare.com` URL pointed at a local port.


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared


### Start the tunnel and read its URL with a real timeout

`readline()` on its own can block forever if `cloudflared` never prints anything
(crashes silently, no output at all) — a plain `while time.time() < deadline`
around it does NOT actually enforce that deadline in that case, since the blocked
`readline()` call itself has no timeout. `select.select(...)` is used here so the
30s deadline is genuinely enforced either way.


In [ ]:
import re
import select

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:9000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    remaining = deadline - time.time()
    ready, _, _ = select.select([tunnel_proc.stdout], [], [], max(remaining, 0))
    if not ready:
        break  # genuinely timed out waiting for output
    line = tunnel_proc.stdout.readline()
    if not line:
        break  # process closed its output (crashed or exited)
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("Router is live at:", public_url)
    print()
    print("Put this in ty's .env:")
    print(f"  API_URL={public_url}/v1")
    print("  API_KEY=colab-local")
    print("  AI_MODEL=qwen3.5-9b   # default. or: llama-3.2-3b for the fast/small one")
    print()
    print("Same URL both times — only AI_MODEL changes which model answers.")
else:
    print("Tunnel URL not found within 30s. Check tunnel_proc output manually, e.g.:")
    print("  print(tunnel_proc.stdout.read())")


## 7. Switching models later

No new URL, no re-running cells — just edit `.env`:
```
AI_MODEL=llama-3.2-3b
```
and restart `ty`. The router reads that string straight out of your request's
`model` field and picks the matching backend. Leaving `AI_MODEL` unset or wrong
falls back to `qwen3.5-9b` (the router's `DEFAULT_MODEL`).


## 8. Stop everything (run before closing the notebook)

In [ ]:
tunnel_proc.terminate()
router_proc.terminate()
for logical_name, port, proc in model_backends:
    proc.terminate()
    print(f"stopped {logical_name}")
print("stopped router and tunnel")
